# GenAI-Traces: RAG Pipeline Tracing

This notebook demonstrates the RAG (Retrieval-Augmented Generation) pipeline tracing capabilities of GenAI-Traces.

## Features Covered
1. Basic RAG tracing with `trace_rag` context manager
2. Recording retrieval results from vector databases
3. Recording embeddings and reranking steps
4. Context assembly tracking
5. Generation recording with groundedness computation
6. Citation tracking
7. Feedback recording
8. Async RAG tracing

In [1]:
import sys
sys.path.insert(0, '..')

from genai_traces import init_tracer, get_tracer
from genai_traces.exporters import ConsoleExporter
from genai_traces.instrumentation import trace_rag, trace_rag_async, RAGTrace, ChunkRecord

# Initialize tracer with console output
tracer = init_tracer(
    service_name="rag-demo",
    environment="notebook",
    exporters=[ConsoleExporter(pretty=True, include_attributes=True)]
)

print("GenAI-Traces RAG Tracing initialized!")

GenAI-Traces RAG Tracing initialized!


## 1. Basic RAG Tracing

The `trace_rag` context manager provides a simple way to trace RAG pipelines.

In [2]:
# Simulated vector database search results
def mock_vector_search(query, top_k=5):
    """Simulate a vector database search."""
    return [
        {
            "id": "doc1_chunk1",
            "content": "Python is a high-level programming language known for its simplicity and readability.",
            "score": 0.92,
            "doc_id": "python_guide",
            "title": "Python Programming Guide",
            "page": 1
        },
        {
            "id": "doc1_chunk2",
            "content": "Python supports multiple programming paradigms including procedural, object-oriented, and functional programming.",
            "score": 0.88,
            "doc_id": "python_guide",
            "title": "Python Programming Guide",
            "page": 2
        },
        {
            "id": "doc2_chunk1",
            "content": "Python was created by Guido van Rossum and first released in 1991.",
            "score": 0.85,
            "doc_id": "python_history",
            "title": "History of Python",
            "page": 1
        },
    ]

# Simulated LLM response
def mock_llm_generate(context, query):
    """Simulate an LLM response."""
    return "Python is a high-level programming language created by Guido van Rossum in 1991. It is known for its simplicity and readability, and supports multiple programming paradigms."

# Basic RAG pipeline with tracing
user_query = "What is Python and who created it?"

with trace_rag(name="python_qa", query=user_query) as rag:
    # Step 1: Retrieve relevant chunks
    chunks = mock_vector_search(user_query, top_k=5)
    rag.record_retrieval(chunks, vector_db="mock_db", top_k=5)
    
    # Step 2: Build context from chunks
    context = "\n\n".join([c["content"] for c in chunks])
    rag.record_context_assembly(context, token_count=len(context.split()))
    
    # Step 3: Generate response
    response = mock_llm_generate(context, user_query)
    rag.record_generation(response, context_used=True)

print(f"\nRAG Summary: {rag.get_summary()}")

[SPAN] python_qa (rag_pipeline) - ok
{
  "trace_id": "b83a23ca737d4e5698bbc987b59cdb69",
  "span_id": "5a0d6c78a16b3134",
  "parent_span_id": null,
  "root_span_id": "5a0d6c78a16b3134",
  "name": "python_qa",
  "span_type": "rag_pipeline",
  "start_time": "2026-04-04T19:52:02.011854",
  "end_time": "2026-04-04T19:52:02.011854",
  "duration_ms": 0.0,
  "status": "ok",
  "status_message": null,
  "attributes": {
    "service.name": "rag-demo",
    "service.environment": "notebook",
    "service.version": "0.0.0",
    "rag.query": "What is Python and who created it?",
    "rag.chunk_count": 3,
    "rag.top_score": 0.92,
    "rag.avg_score": 0.8833333333333333,
    "rag.min_score": 0.85,
    "rag.source_docs": [
      "python_guide",
      "python_history"
    ],
    "rag.unique_sources": 2,
    "rag.vector_db": "mock_db",
    "rag.top_k": 5,
    "rag.context.char_count": 268,
    "rag.context.token_count": 35,
    "rag.context.truncated": false,
    "rag.context_used": true,
    "rag.grou

## 2. Recording Embeddings

Track the query embedding step in your RAG pipeline.

In [3]:
import time

def mock_embed(text, model="text-embedding-ada-002"):
    """Simulate embedding generation."""
    time.sleep(0.05)  # Simulate API latency
    return [0.1] * 1536  # Mock embedding vector

user_query = "How do I install Python packages?"

with trace_rag(name="package_install_qa", query=user_query) as rag:
    # Step 1: Embed the query
    start = time.time()
    embedding = mock_embed(user_query)
    embed_time = (time.time() - start) * 1000
    
    rag.record_embedding(
        embedding=embedding,
        model="text-embedding-ada-002",
        time_ms=embed_time
    )
    
    # Step 2: Search with embedding
    chunks = [
        {"id": "pip1", "content": "Use pip install package_name to install packages.", "score": 0.95},
        {"id": "pip2", "content": "Create a requirements.txt file to manage dependencies.", "score": 0.88},
    ]
    rag.record_retrieval(chunks, time_ms=15.0)
    
    # Step 3: Generate
    response = "To install Python packages, use pip install package_name. You can also create a requirements.txt file."
    rag.record_generation(response)

print(f"Embedding recorded with {len(embedding)} dimensions")

[SPAN] package_install_qa (rag_pipeline) - ok
{
  "trace_id": "45ed31810cbe4d409db01e6b845552ff",
  "span_id": "5a0d9a94f723cded",
  "parent_span_id": null,
  "root_span_id": "5a0d9a94f723cded",
  "name": "package_install_qa",
  "span_type": "rag_pipeline",
  "start_time": "2026-04-04T19:52:13.815914",
  "end_time": "2026-04-04T19:52:13.866517",
  "duration_ms": 50.603,
  "status": "ok",
  "status_message": null,
  "attributes": {
    "service.name": "rag-demo",
    "service.environment": "notebook",
    "service.version": "0.0.0",
    "rag.query": "How do I install Python packages?",
    "rag.embedding.model": "text-embedding-ada-002",
    "rag.embedding.dimensions": 1536,
    "rag.embedding.time_ms": 50.603628158569336,
    "rag.chunk_count": 2,
    "rag.top_score": 0.95,
    "rag.avg_score": 0.915,
    "rag.min_score": 0.88,
    "rag.retrieval.time_ms": 15.0,
    "rag.context_used": true,
    "rag.grounded": true,
    "eval.groundedness": 0.75,
    "rag.groundedness_score": 0.375,
 

## 3. Reranking Support

Track reranking steps when using a cross-encoder or other reranker.

In [4]:
def mock_rerank(query, chunks, model="cross-encoder/ms-marco-MiniLM-L-6-v2"):
    """Simulate reranking with a cross-encoder."""
    # Simulate reranking - reverse order and adjust scores
    reranked = []
    for i, chunk in enumerate(reversed(chunks)):
        reranked.append({
            **chunk,
            "score": 0.95 - (i * 0.05)  # New scores from reranker
        })
    return reranked

user_query = "What are Python decorators?"

with trace_rag(name="decorator_qa", query=user_query) as rag:
    # Initial retrieval
    initial_chunks = [
        {"id": "dec1", "content": "Decorators are a way to modify functions.", "score": 0.75},
        {"id": "dec2", "content": "The @decorator syntax is syntactic sugar.", "score": 0.80},
        {"id": "dec3", "content": "Decorators wrap a function to extend its behavior.", "score": 0.78},
    ]
    rag.record_retrieval(initial_chunks)
    
    # Reranking step
    reranked_chunks = mock_rerank(user_query, initial_chunks)
    rag.record_reranking(
        reranked_chunks,
        reranker_model="cross-encoder/ms-marco-MiniLM-L-6-v2",
        time_ms=25.0
    )
    
    # Generate with reranked context
    response = "Decorators wrap a function to extend its behavior using the @decorator syntax."
    rag.record_generation(response)

print("Reranking recorded successfully")

[SPAN] decorator_qa (rag_pipeline) - ok
{
  "trace_id": "7a68401dd29b4326891593ccf7aa12e5",
  "span_id": "5a0de7216d83a283",
  "parent_span_id": null,
  "root_span_id": "5a0de7216d83a283",
  "name": "decorator_qa",
  "span_type": "rag_pipeline",
  "start_time": "2026-04-04T19:52:33.412036",
  "end_time": "2026-04-04T19:52:33.412036",
  "duration_ms": 0.0,
  "status": "ok",
  "status_message": null,
  "attributes": {
    "service.name": "rag-demo",
    "service.environment": "notebook",
    "service.version": "0.0.0",
    "rag.query": "What are Python decorators?",
    "rag.chunk_count": 3,
    "rag.top_score": 0.95,
    "rag.avg_score": 0.8999999999999999,
    "rag.min_score": 0.85,
    "rag.reranker.model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "rag.reranker.enabled": true,
    "rag.reranker.time_ms": 25.0,
    "rag.reranker.order_changed": true,
    "rag.context_used": true,
    "rag.grounded": true,
    "eval.groundedness": 0.9090909090909091,
    "rag.groundedness_score": 0.4

## 4. Groundedness Computation

The RAG tracer automatically computes a groundedness score based on word overlap between the response and retrieved chunks.

In [5]:
# Example with high groundedness (response uses words from chunks)
with trace_rag(name="grounded_example", query="What is machine learning?") as rag:
    chunks = [
        {
            "id": "ml1",
            "content": "Machine learning is a subset of artificial intelligence that enables systems to learn from data.",
            "score": 0.95
        },
        {
            "id": "ml2",
            "content": "ML algorithms improve through experience without being explicitly programmed.",
            "score": 0.88
        }
    ]
    rag.record_retrieval(chunks)
    
    # Response that uses words from the chunks (high groundedness)
    grounded_response = "Machine learning is a subset of artificial intelligence that enables systems to learn from data and improve through experience."
    rag.record_generation(grounded_response)
    
    print(f"Grounded response - Score: {rag.span.get_attribute('rag.groundedness_score'):.2f}")
    print(f"Is grounded: {rag.span.get_attribute('rag.grounded')}")

# Example with low groundedness (response doesn't use chunk content)
with trace_rag(name="ungrounded_example", query="What is machine learning?") as rag:
    rag.record_retrieval(chunks)  # Same chunks
    
    # Response that doesn't use words from chunks (low groundedness)
    ungrounded_response = "Cats are wonderful pets that bring joy to many households around the world."
    rag.record_generation(ungrounded_response)
    
    print(f"\nUngrounded response - Score: {rag.span.get_attribute('rag.groundedness_score'):.2f}")
    print(f"Is grounded: {rag.span.get_attribute('rag.grounded')}")

Grounded response - Score: 0.63
Is grounded: True
[SPAN] grounded_example (rag_pipeline) - ok
{
  "trace_id": "ae52d97b78614cacbebeff4c069600e7",
  "span_id": "5a0e03ed25e04d5a",
  "parent_span_id": null,
  "root_span_id": "5a0e03ed25e04d5a",
  "name": "grounded_example",
  "span_type": "rag_pipeline",
  "start_time": "2026-04-04T19:52:40.784395",
  "end_time": "2026-04-04T19:52:40.784395",
  "duration_ms": 0.0,
  "status": "ok",
  "status_message": null,
  "attributes": {
    "service.name": "rag-demo",
    "service.environment": "notebook",
    "service.version": "0.0.0",
    "rag.query": "What is machine learning?",
    "rag.chunk_count": 2,
    "rag.top_score": 0.95,
    "rag.avg_score": 0.915,
    "rag.min_score": 0.88,
    "rag.context_used": true,
    "rag.grounded": true,
    "eval.groundedness": 1.0,
    "rag.groundedness_score": 0.631578947368421
  },
  "events": [
    {
      "name": "retrieval_complete",
      "timestamp": "2026-04-04T19:52:40.784395",
      "attributes": {

## 5. Citation Tracking

Track which chunks were cited in the response.

In [6]:
with trace_rag(name="citation_example", query="Explain Python's GIL") as rag:
    chunks = [
        {"id": "gil1", "content": "The GIL is a mutex that protects access to Python objects.", "score": 0.92},
        {"id": "gil2", "content": "The GIL prevents multiple threads from executing Python bytecode simultaneously.", "score": 0.89},
        {"id": "gil3", "content": "Multiprocessing can be used to bypass the GIL.", "score": 0.85},
    ]
    rag.record_retrieval(chunks)
    
    # Response with citations
    response = "The GIL (Global Interpreter Lock) is a mutex that protects Python objects [1]. It prevents multiple threads from executing bytecode simultaneously [2]. To bypass this, use multiprocessing [3]."
    rag.record_generation(response)
    
    # Record which chunks were cited
    citations = [
        {"chunk_id": "gil1", "position": 1, "text": "[1]"},
        {"chunk_id": "gil2", "position": 2, "text": "[2]"},
        {"chunk_id": "gil3", "position": 3, "text": "[3]"},
    ]
    rag.record_citation(citations)

print(f"Citation count: {rag.span.get_attribute('rag.citation_count')}")
print(f"Citation coverage: {rag.span.get_attribute('rag.citation_coverage'):.0%}")

[SPAN] citation_example (rag_pipeline) - ok
{
  "trace_id": "afb08d4a87aa4b21b58aeb065df72d98",
  "span_id": "5a0e155da3f30a41",
  "parent_span_id": null,
  "root_span_id": "5a0e155da3f30a41",
  "name": "citation_example",
  "span_type": "rag_pipeline",
  "start_time": "2026-04-04T19:52:45.248002",
  "end_time": "2026-04-04T19:52:45.248002",
  "duration_ms": 0.0,
  "status": "ok",
  "status_message": null,
  "attributes": {
    "service.name": "rag-demo",
    "service.environment": "notebook",
    "service.version": "0.0.0",
    "rag.query": "Explain Python's GIL",
    "rag.chunk_count": 3,
    "rag.top_score": 0.92,
    "rag.avg_score": 0.8866666666666667,
    "rag.min_score": 0.85,
    "rag.context_used": true,
    "rag.grounded": true,
    "eval.groundedness": 1.0,
    "rag.groundedness_score": 0.7058823529411765,
    "rag.citation_count": 3,
    "rag.citations": [
      {
        "chunk_id": "gil1",
        "position": 1,
        "text": "[1]"
      },
      {
        "chunk_id": "

## 6. Feedback Recording

Record user feedback on RAG responses for quality monitoring.

In [7]:
with trace_rag(name="feedback_example", query="How to debug Python code?") as rag:
    chunks = [
        {"id": "debug1", "content": "Use print statements or logging for basic debugging.", "score": 0.90},
        {"id": "debug2", "content": "Python debugger (pdb) allows step-by-step execution.", "score": 0.88},
    ]
    rag.record_retrieval(chunks)
    
    response = "Debug Python using print statements, logging, or pdb for step-by-step execution."
    rag.record_generation(response)
    
    # Record user feedback
    rag.record_feedback(
        helpful=True,
        accurate=True,
        score=5,
        comment="Very helpful, covered all the basics!"
    )

print(f"Feedback recorded:")
print(f"  Helpful: {rag.span.get_attribute('rag.feedback.helpful')}")
print(f"  Score: {rag.span.get_attribute('rag.feedback.score')}")

[SPAN] feedback_example (rag_pipeline) - ok
{
  "trace_id": "d6cac31c624e46f1994f613785c5a33f",
  "span_id": "5a0e2ae7124f3e03",
  "parent_span_id": null,
  "root_span_id": "5a0e2ae7124f3e03",
  "name": "feedback_example",
  "span_type": "rag_pipeline",
  "start_time": "2026-04-04T19:52:50.762685",
  "end_time": "2026-04-04T19:52:50.763684",
  "duration_ms": 0.9990000000000001,
  "status": "ok",
  "status_message": null,
  "attributes": {
    "service.name": "rag-demo",
    "service.environment": "notebook",
    "service.version": "0.0.0",
    "rag.query": "How to debug Python code?",
    "rag.chunk_count": 2,
    "rag.top_score": 0.9,
    "rag.avg_score": 0.89,
    "rag.min_score": 0.88,
    "rag.context_used": true,
    "rag.grounded": true,
    "eval.groundedness": 0.7272727272727273,
    "rag.groundedness_score": 0.36363636363636365,
    "rag.feedback.helpful": true,
    "rag.feedback.accurate": true,
    "rag.feedback.score": 5,
    "rag.feedback.comment": "Very helpful, covered a

## 7. Context Assembly Tracking

Track how context is assembled from chunks, including truncation.

In [8]:
def assemble_context(chunks, max_tokens=500):
    """Assemble context from chunks with token limit."""
    context_parts = []
    total_tokens = 0
    truncated = False
    
    for chunk in chunks:
        chunk_tokens = len(chunk["content"].split())
        if total_tokens + chunk_tokens > max_tokens:
            truncated = True
            break
        context_parts.append(chunk["content"])
        total_tokens += chunk_tokens
    
    return "\n\n".join(context_parts), total_tokens, truncated

with trace_rag(name="context_assembly_demo", query="Explain Python async/await") as rag:
    # Retrieve many chunks
    chunks = [
        {"id": f"async{i}", "content": f"Async chunk {i}: " + "word " * 100, "score": 0.9 - i*0.05}
        for i in range(10)
    ]
    rag.record_retrieval(chunks)
    
    # Assemble context with token limit
    context, token_count, truncated = assemble_context(chunks, max_tokens=300)
    rag.record_context_assembly(
        context=context,
        token_count=token_count,
        max_tokens=300,
        truncated=truncated
    )
    
    response = "Python async/await enables asynchronous programming."
    rag.record_generation(response)

print(f"Context tokens: {rag.span.get_attribute('rag.context.token_count')}")
print(f"Truncated: {rag.span.get_attribute('rag.context.truncated')}")
print(f"Utilization: {rag.span.get_attribute('rag.context.utilization'):.0%}")

[SPAN] context_assembly_demo (rag_pipeline) - ok
{
  "trace_id": "26f5579fca4c4ff297857ef3816cc2a5",
  "span_id": "5a0e3f33609a323a",
  "parent_span_id": null,
  "root_span_id": "5a0e3f33609a323a",
  "name": "context_assembly_demo",
  "span_type": "rag_pipeline",
  "start_time": "2026-04-04T19:52:55.958832",
  "end_time": "2026-04-04T19:52:55.958832",
  "duration_ms": 0.0,
  "status": "ok",
  "status_message": null,
  "attributes": {
    "service.name": "rag-demo",
    "service.environment": "notebook",
    "service.version": "0.0.0",
    "rag.query": "Explain Python async/await",
    "rag.chunk_count": 10,
    "rag.top_score": 0.9,
    "rag.avg_score": 0.675,
    "rag.min_score": 0.45,
    "rag.context.char_count": 1032,
    "rag.context.token_count": 206,
    "rag.context.truncated": true,
    "rag.context.max_tokens": 300,
    "rag.context.utilization": 0.6866666666666666,
    "rag.context_used": true,
    "rag.grounded": false,
    "eval.groundedness": 0.0,
    "rag.groundedness_sc

## 8. Async RAG Tracing

Use `trace_rag_async` for asynchronous RAG pipelines.

In [ ]:
import asyncio

async def async_vector_search(query):
    """Simulate async vector search."""
    await asyncio.sleep(0.1)  # Simulate network latency
    return [
        {"id": "async1", "content": "Asyncio is Python's async framework.", "score": 0.92},
        {"id": "async2", "content": "Use async/await for concurrent I/O.", "score": 0.88},
    ]

async def async_llm_generate(context, query):
    """Simulate async LLM call."""
    await asyncio.sleep(0.1)
    return "Asyncio is Python's async framework. Use async/await for concurrent I/O operations."

async def async_rag_pipeline():
    query = "How does Python asyncio work?"
    
    async with trace_rag_async(name="async_rag_demo", query=query) as rag:
        # Async retrieval
        chunks = await async_vector_search(query)
        rag.record_retrieval(chunks)
        
        # Async generation
        context = "\n".join([c["content"] for c in chunks])
        response = await async_llm_generate(context, query)
        rag.record_generation(response)
        
        return rag.get_summary()

# Run async pipeline
summary = await async_rag_pipeline()
print(f"Async RAG Summary: {summary}")

## 9. Complete RAG Pipeline Example

A comprehensive example showing all RAG tracing features together.

In [ ]:
import time

def complete_rag_pipeline(user_query: str):
    """Complete RAG pipeline with full tracing."""
    
    with trace_rag(
        name="complete_rag_demo",
        query=user_query,
        user_id="demo_user_123",
        session_id="session_456"
    ) as rag:
        
        # Step 1: Embed query
        start = time.time()
        embedding = [0.1] * 1536  # Mock embedding
        rag.record_embedding(
            embedding=embedding,
            model="text-embedding-3-small",
            time_ms=(time.time() - start) * 1000
        )
        
        # Step 2: Vector search
        start = time.time()
        initial_chunks = [
            {
                "id": "chunk_001",
                "content": "FastAPI is a modern Python web framework for building APIs.",
                "score": 0.89,
                "doc_id": "fastapi_docs",
                "title": "FastAPI Documentation",
                "page": 1
            },
            {
                "id": "chunk_002",
                "content": "FastAPI uses Python type hints for automatic validation.",
                "score": 0.85,
                "doc_id": "fastapi_docs",
                "title": "FastAPI Documentation",
                "page": 5
            },
            {
                "id": "chunk_003",
                "content": "Pydantic models define request and response schemas.",
                "score": 0.82,
                "doc_id": "pydantic_guide",
                "title": "Pydantic Guide",
                "page": 1
            },
        ]
        rag.record_retrieval(
            initial_chunks,
            vector_db="pinecone",
            top_k=10,
            time_ms=(time.time() - start) * 1000
        )
        
        # Step 3: Rerank
        start = time.time()
        reranked_chunks = [
            {**initial_chunks[0], "score": 0.95},
            {**initial_chunks[2], "score": 0.90},
            {**initial_chunks[1], "score": 0.85},
        ]
        rag.record_reranking(
            reranked_chunks,
            reranker_model="cohere-rerank-v3",
            time_ms=(time.time() - start) * 1000
        )
        
        # Step 4: Assemble context
        context = "\n\n".join([c["content"] for c in reranked_chunks])
        rag.record_context_assembly(
            context=context,
            token_count=len(context.split()),
            max_tokens=4000
        )
        
        # Step 5: Generate response
        start = time.time()
        response = "FastAPI is a modern Python web framework for building APIs. It uses Python type hints for automatic validation and Pydantic models for request/response schemas [1][2][3]."
        rag.record_generation(
            response,
            context_used=True,
            time_ms=(time.time() - start) * 1000
        )
        
        # Step 6: Record citations
        rag.record_citation([
            {"chunk_id": "chunk_001", "position": 1},
            {"chunk_id": "chunk_002", "position": 2},
            {"chunk_id": "chunk_003", "position": 3},
        ])
        
        return response, rag.get_summary()

# Run the complete pipeline
response, summary = complete_rag_pipeline("What is FastAPI and how does it handle validation?")

print("\n" + "="*60)
print("COMPLETE RAG PIPELINE RESULTS")
print("="*60)
print(f"\nResponse: {response}")
print(f"\nSummary:")
for key, value in summary.items():
    print(f"  {key}: {value}")

## 10. ChunkRecord Details

Explore the `ChunkRecord` dataclass for detailed chunk information.

In [ ]:
# Create ChunkRecord directly
chunk = ChunkRecord(
    chunk_id="doc1_p5_c3",
    content="This is the content of the chunk with important information about the topic.",
    score=0.92,
    source_doc_id="document_001",
    source_doc_page=5,
    source_doc_title="Technical Documentation",
    fetch_timestamp="2024-01-15T10:30:00Z",
    metadata={"section": "Introduction", "author": "John Doe"}
)

print("ChunkRecord Details:")
print(f"  ID: {chunk.chunk_id}")
print(f"  Score: {chunk.score}")
print(f"  Source: {chunk.source_doc_title} (page {chunk.source_doc_page})")
print(f"  Content preview: {chunk.content[:50]}...")
print(f"\nAs dict: {chunk.to_dict()}")

## Summary

This notebook demonstrated the RAG pipeline tracing capabilities:

1. **Basic Tracing**: Use `trace_rag` context manager to wrap RAG pipelines
2. **Embedding Recording**: Track query embedding with model and timing info
3. **Retrieval Recording**: Capture chunks with scores, sources, and metadata
4. **Reranking Support**: Track reranking steps with model and order changes
5. **Groundedness**: Automatic computation of response groundedness
6. **Citations**: Track which chunks were cited in responses
7. **Feedback**: Record user feedback for quality monitoring
8. **Context Assembly**: Track context building with token limits
9. **Async Support**: Full async pipeline tracing with `trace_rag_async`

These features provide complete visibility into RAG pipeline performance, quality, and user satisfaction.